# Stage 4: Cleaning, encoding, and scaling

This notebook makes preprocessing decisions using the training split only. The same pipeline will be used for training and serving.

In [3]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent
if not (repo_root / "src").exists():
    repo_root = Path("/Users/leshakamadara/bank-LM-prediction")
sys.path.insert(0, str(repo_root))

from src.data import get_split
from src.features import CATEGORICAL, NUMERIC
from src.pipeline import build

X_train, X_test, y_train, y_test = get_split()
train_data = X_train.copy()
train_data["y"] = y_train
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 36168
Test rows: 9043


## 1. Missing values and the `unknown` category

In [4]:
missing_counts = X_train.isna().sum()
print("Missing values by column:")
display(missing_counts[missing_counts > 0].to_frame("count"))

unknown_rows = train_data[CATEGORICAL].eq("unknown").any(axis=1)
unknown_yes_rate = train_data.loc[unknown_rows, "y"].mean()
unknown_count = unknown_rows.sum()
print(f"Rows with at least one 'unknown' category: {unknown_count}")
print(f"Yes-rate for those rows: {unknown_yes_rate:.3f}")

unknown_by_column = []
for column in CATEGORICAL:
    rows = train_data[column] == "unknown"
    unknown_by_column.append((column, rows.sum(), train_data.loc[rows, "y"].mean()))
display(pd.DataFrame(unknown_by_column, columns=["column", "unknown_count", "yes_rate"]))

Missing values by column:


,count


Rows with at least one 'unknown' category: 29917
Yes-rate for those rows: 0.094


,column,unknown_count,yes_rate
0,job,234,0.119658
1,marital,0,NaN
2,education,1482,0.132254
3,default,0,NaN
4,housing,0,NaN
5,loan,0,NaN
6,contact,10386,0.041113
7,month,0,NaN
8,poutcome,29589,0.092230


## Decision and reason

Keep `unknown` as its own category because the missingness can carry signal about how the bank knows the client. One-hot encoding will give it its own indicator instead of replacing it with a made-up value.

## 2. Duplicate rows

In [ ]:
duplicate_count = X_train.duplicated().sum()
print(f"Duplicate feature rows in training data: {duplicate_count}")
print(f"Duplicate percentage: {duplicate_count / len(X_train) * 100:.2f}%")

## Decision and reason

Count duplicates before modelling. We keep them for now because repeated client records can represent real repeated campaign observations; removing them without checking their meaning could remove valid evidence.

## 3. Outliers and the IQR rule

In [ ]:
outlier_columns = ["balance", "campaign", "previous", "pdays"]
iqr_rows = []
outlier_masks = []
for column in outlier_columns:
    first_quartile = X_train[column].quantile(0.25)
    third_quartile = X_train[column].quantile(0.75)
    iqr = third_quartile - first_quartile
    lower_bound = first_quartile - 1.5 * iqr
    upper_bound = third_quartile + 1.5 * iqr
    mask = (X_train[column] < lower_bound) | (X_train[column] > upper_bound)
    outlier_masks.append(mask)
    iqr_rows.append((column, lower_bound, upper_bound, mask.sum(), y_train[mask].mean()))

iqr_table = pd.DataFrame(
    iqr_rows,
    columns=["column", "lower_bound", "upper_bound", "rows_flagged", "yes_rate"],
)
display(iqr_table)

any_outlier = pd.concat(outlier_masks, axis=1).any(axis=1)
print(f"Rows flagged by at least one IQR rule: {any_outlier.sum()}")
print(f"Yes-rate of rows flagged by any IQR rule: {y_train[any_outlier].mean():.3f}")

## Decision and reason

Keep the outliers. Tree models are generally insensitive to feature scale and extreme numeric values, and the pipeline can use a log feature for the strongly skewed `balance` variable later. The IQR counts are useful diagnostics, but are not enough reason to delete potentially real clients.

## 4. Encoding categorical variables

## Decision and reason

Use one-hot encoding because the categorical values do not have a natural numeric order. Set `handle_unknown="ignore"` so the API returns a prediction instead of crashing when a new category appears.

## 5. Scaling numeric variables

## Decision and reason

Use `StandardScaler` because distance-based kNN needs comparable numeric scales. Scaling is harmless for tree models, so one shared preprocessing pipeline can support both families.

## 6. Demonstrate the shared pipeline

In [ ]:
model = build(LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

features_train = model.named_steps["features"].transform(X_train)
features_test = model.named_steps["features"].transform(X_test)
prep = model.named_steps["prep"]
processed_train = prep.transform(features_train)
processed_test = prep.transform(features_test)
feature_names = prep.get_feature_names_out()

print(f"Processed train shape: {processed_train.shape}")
print(f"Processed test shape: {processed_test.shape}")
print(f"Processed column count: {len(feature_names)}")
print("Processed column names:")
print(feature_names.tolist())

In [ ]:
numeric_count = len(NUMERIC)
train_numeric_means = processed_train[:, :numeric_count].mean(axis=0)
test_numeric_means = processed_test[:, :numeric_count].mean(axis=0)
scaling_check = pd.DataFrame({
    "column": NUMERIC,
    "train_mean": train_numeric_means,
    "test_mean": test_numeric_means,
})
display(scaling_check)
print("Train means are approximately zero because the scaler was fitted on X_train.")
print("Test means are not expected to be exactly zero because X_test was not used to fit preprocessing.")

## Decision and reason

Fit the complete pipeline on `X_train` only, then transform both splits. The train numeric means are about zero, while test means are not exactly zero. That difference is correct: it shows that the scaler learned its statistics from training data and did not leak information from the test set.

## 7. Preprocessing decisions summary

In [8]:
import pandas as pd

decisions = pd.DataFrame([
    ("Missing categorical values", "Keep 'unknown' as its own category", "Unknown status can carry signal; its yes-rate is measured above."),
    ("Duplicate rows", "Count and keep for now", "Repeated records may represent valid campaign observations."),
    ("Outliers", "Keep balance, campaign, previous, and pdays outliers", "Tree models are insensitive; balance can receive a log feature later."),
    ("Categorical encoding", "One-hot encoding with handle_unknown=ignore", "Avoids imposing order and prevents API crashes on new values."),
    ("Numeric scaling", "StandardScaler fitted on training data", "Needed for kNN and harmless for tree models."),
], columns=["Decision area", "Choice", "Reason"])
display(decisions)

,Decision area,Choice,Reason
0,Missing categorical values,Keep 'unknown' as its own category,Unknown status can carry signal; its yes-rate ...
1,Duplicate rows,Count and keep for now,Repeated records may represent valid campaign ...
2,Outliers,"Keep balance, campaign, previous, and pdays ou...",Tree models are insensitive; balance can recei...
3,Categorical encoding,One-hot encoding with handle_unknown=ignore,Avoids imposing order and prevents API crashes...
4,Numeric scaling,StandardScaler fitted on training data,Needed for kNN and harmless for tree models.
